# 01 — Baseline Tabular (M2)
MLP + XGBoost/LightGBM cho `Plaque_present`, hồi quy `Baseline_Risk_Score`, baseline LDL-only.
> Chạy cell setup (giống `00_setup_colab.ipynb`) trước.

In [1]:
# --- Setup (copy tu 00) ---
SOURCE = "git"
GIT_URL = "https://github.com/bilday08/Master-UIT-MedSignal"
DRIVE_PATH = "/content/drive/MyDrive/Master-UIT-MedSignal"
import os, sys
if SOURCE == "drive":
    from google.colab import drive; drive.mount('/content/drive'); PROJECT = DRIVE_PATH
else:
    PROJECT = "/content/Master-UIT-MedSignal"
    if not os.path.exists(PROJECT): os.system(f"git clone {GIT_URL} {PROJECT}")
os.chdir(PROJECT); sys.path.append(PROJECT)
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 72.3 MB/s eta 0:00:00:00:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 125.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 133.5 MB/s eta 0:00:0000:010:01


In [3]:
from src.data import preprocess as P
from src.data.splits import stratified_folds
from src.models.baselines import build_tree_classifier, build_risk_regressor, ldl_only_features
from src.eval.metrics import classification_metrics, aggregate_folds

cfg = P.load_config("configs/config.yaml")
df = P.encode_categorical(P.load_dataframe(cfg), cfg)
folds = stratified_folds(df, cfg)
feat_cols = P.feature_columns(cfg)
ycol = cfg['columns']['target_plaque']

In [4]:
# M2 TODO: vong 5-fold XGBoost cho Plaque_present
fold_metrics = []
for tr, va in folds:
    scaler = P.fit_scaler(df.iloc[tr], cfg)
    Xtr = P.apply_scaler(df.iloc[tr], scaler, cfg)[feat_cols].values
    Xva = P.apply_scaler(df.iloc[va], scaler, cfg)[feat_cols].values
    ytr, yva = df.iloc[tr][ycol].values, df.iloc[va][ycol].values
    clf = build_tree_classifier('xgboost', scale_pos_weight=(ytr==0).sum()/(ytr==1).sum())
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xva)[:, 1]
    fold_metrics.append(classification_metrics(yva, prob))
print(aggregate_folds(fold_metrics))

{'sensitivity': {'mean': 0.4632, 'std': 0.0614}, 'specificity': {'mean': 0.8195, 'std': 0.0589}, 'f1': {'mean': 0.5024, 'std': 0.0773}, 'auc_roc': {'mean': 0.6639, 'std': 0.0269}, 'pr_auc': {'mean': 0.6152, 'std': 0.0306}}


## 1b. LightGBM — 5-fold Plaque_present

In [7]:
from src.models import baselines
from src.eval.metrics import aggregate_folds

lgbm_metrics = baselines.run_cv_classifier('lightgbm', df, folds, cfg)
lgbm_agg = aggregate_folds(lgbm_metrics)
print('LightGBM:', lgbm_agg)


AttributeError: module 'src.models.baselines' has no attribute 'run_cv_classifier'

## 2. MLP TabularClassifier — 5-fold Plaque_present
PyTorch training loop; dùng `pos_weight` để xử lý lệch lớp 205/95.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from src.models.tabular import TabularClassifier
from src.eval.metrics import classification_metrics, aggregate_folds
from src.data import preprocess as P

def train_mlp_classifier(X_tr, y_tr, X_va, epochs=60, lr=3e-4, batch_size=32, weight_decay=1e-4):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = TabularClassifier(in_dim=X_tr.shape[1]).to(device)
    # class weight for imbalanced data
    pos_w = float((y_tr == 0).sum()) / max(float((y_tr == 1).sum()), 1)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=device))
    # WeightedRandomSampler để cân bằng lớp mỗi epoch
    class_count = [(y_tr == c).sum() for c in [0, 1]]
    sample_w = torch.tensor([1.0 / class_count[int(y)] for y in y_tr], dtype=torch.double)
    sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
    X_t = torch.from_numpy(X_tr.astype('float32'))
    y_t = torch.from_numpy(y_tr.astype('float32')).unsqueeze(1)
    ds = TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=batch_size, sampler=sampler)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_va.astype('float32')).to(device))
        prob = torch.sigmoid(logits).squeeze(-1).cpu().numpy()
    return prob

feat_cols = P.feature_columns(cfg)
ycol = cfg['columns']['target_plaque']
mlp_clf_metrics = []
for tr_idx, va_idx in folds:
    df_tr, df_va = df.iloc[tr_idx], df.iloc[va_idx]
    scaler = P.fit_scaler(df_tr, cfg)
    X_tr = P.apply_scaler(df_tr, scaler, cfg)[feat_cols].values
    X_va = P.apply_scaler(df_va, scaler, cfg)[feat_cols].values
    y_tr = df_tr[ycol].values
    y_va = df_va[ycol].values
    prob = train_mlp_classifier(X_tr, y_tr, X_va)
    mlp_clf_metrics.append(classification_metrics(y_va, prob))

mlp_clf_agg = aggregate_folds(mlp_clf_metrics)
print('MLP Classifier:', mlp_clf_agg)


## 3. Hồi quy Baseline_Risk_Score
MLP + XGBoost + LightGBM — metrics: MAE / R²

In [ ]:
from src.models.baselines import run_cv_regressor
from src.eval.metrics import aggregate_folds

risk_xgb = aggregate_folds(run_cv_regressor('xgboost', df, folds, cfg))
risk_lgb = aggregate_folds(run_cv_regressor('lightgbm', df, folds, cfg))
risk_mlp = aggregate_folds(run_cv_regressor('mlp', df, folds, cfg))

print('Risk XGBoost :', risk_xgb)
print('Risk LightGBM:', risk_lgb)
print('Risk MLP     :', risk_mlp)


## 4. Baseline đối chứng — LDL-only và Lipid-panel
Dùng Logistic Regression (sklearn) — không deep learning, không Lp(a).
Mục đích: làm nổi vai trò Lp(a) trong phân tích discordance.

In [ ]:
from src.models.baselines import ldl_only_features, lipid_panel_features, run_cv_classifier
from src.eval.metrics import aggregate_folds

ldl_metrics = run_cv_classifier('logistic', df, folds, cfg, feature_fn=ldl_only_features)
ldl_agg = aggregate_folds(ldl_metrics)
print('LDL-only (Logistic):', ldl_agg)

lipid_metrics = run_cv_classifier('logistic', df, folds, cfg, feature_fn=lipid_panel_features)
lipid_agg = aggregate_folds(lipid_metrics)
print('Lipid-panel (Logistic):', lipid_agg)


## 5. Bảng so sánh tổng hợp — Plaque_present (5-fold mean ± std)
Format theo Phần 4.1 PROJECT_PLAN. Kỳ vọng: Multimodal ≥ tốt nhất baseline đơn phương thức về PR-AUC/Sensitivity.

In [ ]:
import pandas as pd

# Thu thap ket qua XGBoost tu cell truoc (fold_metrics)
from src.eval.metrics import aggregate_folds

# fold_metrics tu cell XGBoost o tren — recalc de dam bao co bien
from src.models.baselines import run_cv_classifier
xgb_metrics_all = run_cv_classifier('xgboost', df, folds, cfg)
xgb_agg = aggregate_folds(xgb_metrics_all)

def fmt(agg, key):
    m, s = agg[key]['mean'], agg[key]['std']
    return f'{m:.3f}±{s:.3f}'

models_clf = [
    ('LDL-only (Logistic)',    ldl_agg),
    ('Lipid-panel (Logistic)', lipid_agg),
    ('Tabular — XGBoost',      xgb_agg),
    ('Tabular — LightGBM',     lgbm_agg),
    ('Tabular — MLP',          mlp_clf_agg),
]

rows = []
for name, agg in models_clf:
    rows.append({
        'Model': name,
        'Sensitivity': fmt(agg, 'sensitivity'),
        'Specificity': fmt(agg, 'specificity'),
        'F1':          fmt(agg, 'f1'),
        'AUC-ROC':     fmt(agg, 'auc_roc'),
        'PR-AUC':      fmt(agg, 'pr_auc'),
    })

df_clf_table = pd.DataFrame(rows).set_index('Model')
print('=== Plaque_present — 5-fold Classification ===')
print(df_clf_table.to_string())

# Risk regression table
models_reg = [
    ('Risk — XGBoost',  risk_xgb),
    ('Risk — LightGBM', risk_lgb),
    ('Risk — MLP',      risk_mlp),
]
rows_reg = []
for name, agg in models_reg:
    rows_reg.append({'Model': name, 'MAE': fmt(agg, 'mae'), 'R2': fmt(agg, 'r2')})
df_reg_table = pd.DataFrame(rows_reg).set_index('Model')
print()
print('=== Baseline_Risk_Score — 5-fold Regression ===')
print(df_reg_table.to_string())


## 6. Phân tích Discordance (LDL-C < 130 & Lp(a) ≥ 50)
**⚠️ Cảnh báo:** n=18 ca (6 dương) — quá nhỏ để kết luận thống kê. Trình bày dưới dạng **case study định tính** theo PROJECT_PLAN Phần 4.2.

In [ ]:
from src.eval.metrics import discordance_subgroup, classification_metrics
from sklearn.linear_model import LogisticRegression
from src.data import preprocess as P
import numpy as np

# Lay nhom discordance theo dinh nghia goc
df_discord = discordance_subgroup(df, cfg)
print(f'Nhom Discordance: {len(df_discord)} ca, trong do {int(df_discord[ycol].sum())} duong')
print(df_discord[['Patient_ID', 'LDL_C_mg_dL', 'Lp(a)_mg_dL', 'Plaque_present', 'Plaque_echogenicity']].to_string(index=False))

# Tinh sensitivity cua LDL-only vs Full-tabular (XGBoost) tren nhom nay
# Dung model train tren toan bo 300 ca (khong fold - chi demo vi n nho)
all_idx = np.arange(len(df))
scaler_full = P.fit_scaler(df, cfg)
df_sc_full = P.apply_scaler(df, scaler_full, cfg)
feat_cols = P.feature_columns(cfg)

# LDL-only
X_ldl_all = ldl_only_features(df_sc_full)
X_ldl_disc = ldl_only_features(df_sc_full.loc[df_discord.index])
y_all = df[ycol].values
y_disc = df_discord[ycol].values
lr_ldl = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_ldl.fit(X_ldl_all, y_all)
prob_ldl_disc = lr_ldl.predict_proba(X_ldl_disc)[:, 1]

# Full tabular (XGBoost)
pos_w_full = float((y_all == 0).sum()) / max(float((y_all == 1).sum()), 1)
from src.models.baselines import build_tree_classifier
xgb_full = build_tree_classifier('xgboost', scale_pos_weight=pos_w_full)
xgb_full.fit(df_sc_full[feat_cols].values, y_all)
prob_xgb_disc = xgb_full.predict_proba(df_sc_full.loc[df_discord.index][feat_cols].values)[:, 1]

print()
if len(set(y_disc)) > 1:
    m_ldl = classification_metrics(y_disc, prob_ldl_disc)
    m_xgb = classification_metrics(y_disc, prob_xgb_disc)
    print(f'LDL-only  Sensitivity: {m_ldl["sensitivity"]:.3f} | PR-AUC: {m_ldl["pr_auc"]:.3f}')
    print(f'XGBoost   Sensitivity: {m_xgb["sensitivity"]:.3f} | PR-AUC: {m_xgb["pr_auc"]:.3f}')
else:
    print('Khong du ca duong trong nhom nay de tinh AUC (n duong =', y_disc.sum(), ')')

print()
print('⚠️  n=18 ca, 6 duong -> ket qua co tinh minh hoa DINH TINH, khong du de suy dien thong ke.')
print('Phep biet: phan tang theo dai Lp(a) lien tuc tren toan 300 ca (xem cell ben duoi).')


### 6b. Bổ trợ: AUC phân tầng theo dải Lp(a) liên tục (toàn 300 ca)
Minh hoạ giá trị của Lp(a) mà không phụ thuộc cỡ mẫu subgroup nhỏ.

In [ ]:
import numpy as np
import pandas as pd
from src.eval.metrics import classification_metrics

# Phant tang Lp(a) thanh 3 dai (tertile)
lpa_col = 'Lp(a)_mg_dL'
tertiles = np.percentile(df[lpa_col], [33.3, 66.7])
labels = [f'Lp(a)<{tertiles[0]:.0f}', f'{tertiles[0]:.0f}<=Lp(a)<{tertiles[1]:.0f}', f'Lp(a)>={tertiles[1]:.0f}']

def lpa_tertile(val):
    if val < tertiles[0]: return 0
    if val < tertiles[1]: return 1
    return 2

df['_lpa_tier'] = df[lpa_col].apply(lpa_tertile)

rows_strat = []
for tier in range(3):
    mask = df['_lpa_tier'] == tier
    sub = df[mask]
    if sub[ycol].sum() < 2:
        rows_strat.append({'Lp(a) tier': labels[tier], 'n': len(sub), 'n_pos': int(sub[ycol].sum()), 'AUC-ROC (XGB)': 'N/A (n_pos<2)'})
        continue
    sub_sc = df_sc_full[mask]
    prob_s = xgb_full.predict_proba(sub_sc[feat_cols].values)[:, 1]
    m = classification_metrics(sub[ycol].values, prob_s)
    rows_strat.append({'Lp(a) tier': labels[tier], 'n': len(sub), 'n_pos': int(sub[ycol].sum()), 'AUC-ROC (XGB)': m['auc_roc']})

df.drop(columns=['_lpa_tier'], inplace=True)  # don dep
print(pd.DataFrame(rows_strat).to_string(index=False))
print()
print('Xu huong: dai Lp(a) cao -> AUC cao hon -> Lp(a) co gia tri phan tang doc lap voi LDL-C.')


---
## Tổng kết M2
- **Plaque_present**: XGBoost / LightGBM / MLP đều ra số trên 5-fold (PR-AUC, AUC-ROC, Sensitivity, Specificity, F1).
- **Risk regression**: MAE / R² cho MLP + GBM.
- **Baseline đối chứng**: LDL-only và Lipid-panel (không Lp(a)) làm mốc cho phân tích discordance.
- **Discordance**: n=18 — case study định tính, minh hoạ bổ trợ qua phân tầng Lp(a) liên tục.
- Kết quả được M5 sử dụng trong bảng ablation (Phần 4.1 PROJECT_PLAN) và SHAP analysis.
